# Feature Engineering Agrégations

Effectuer des agrégations par IP/utilisateur/session : nombre de destinations uniques, volume de données, durée de session, diversité des actions. Créer un dataframe final de features.

## Importation des librairies

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
chemin_CICIDS = '../Data/cicids_clean.csv'
chemin_UNSW   = '../Data/unsw_clean.csv'
chemin_LOGS   = '../Data/logs_clean.csv'

df_cicids = pd.read_csv(chemin_CICIDS, low_memory=False)
df_unsw   = pd.read_csv(chemin_UNSW,   low_memory=False)
df_logs   = pd.read_csv(chemin_LOGS,   low_memory=False)


## Dataset 1 : CIC-IDS-2017


### 1.1 Agrégation par port de destination

On regroupe tous les flux par port de destination pour obtenir un profil par service réseau car un port qui reçoit un très grand volume de données ou beaucoup de connexions courtes est suspect.

In [6]:
agg_cicids = df_cicids.groupby('Destination Port').agg(
    nb_flux  = ('Destination Port', 'count'),      # combien de connexions vers ce port
    volume_total_octets = ('Total Length of Fwd Packets', 'sum'), # total d'octets envoyés vers ce port
    duree_moy = ('Flow Duration', 'mean'),          # durée moyenne d'une connexion vers ce port
    duree_max = ('Flow Duration', 'max'),           # connexion la plus longue vers ce port
    paquets_moy= ('Total Fwd Packets', 'mean'),      # nombre moyen de paquets par connexion
    taux_attaque = ('Label', lambda x: (x != 'BENIGN').mean()) # pourcentage de connexions malveillantes
)
print(f"Nombre de ports distincts : {len(agg_cicids)}")
print(agg_cicids.nlargest(10, 'nb_flux').round(2))

Nombre de ports distincts : 23944
                  nb_flux  volume_total_octets    duree_moy  duree_max  \
Destination Port                                                         
80                 136556             12269584  17259727.22  119999872   
53                  30302              2378282   1459497.04  119991573   
443                 13111             17921393  30354426.14  119999439   
8080                  510                  791   6177295.37   60532774   
123                   361                21936   7722964.91   66025810   
22                    317               444838    731020.44    1637463   
389                   257               574848  23773119.42  119527967   
88                    170               220138       724.67      10463   
21                    143                15230     79837.56     191917   
137                   136               118952    865034.76   11602681   

                  paquets_moy  taux_attaque  
Destination Port               

Maintenant au lieu d'avoir 223 080 lignes (une par connexion) on a  23 944 lignes donc une par port. Chaque ligne résume tout ce qui s'est passé sur ce portl'agrégation, on révèle 23 944 ports distincts. Le port 80 (HTTP) domine avec 136 556 flux et un volume de 12 millions d'octets et surtout un taux d'attaque de 0.94 cela veut dire que 94% des connexions vers le port 80 sont des attaques DDoS. En agrégeant par port, on identifie immédiatement le port 80 comme la cible principale des attaques DDoS. Tous les autres ports dans le top 10 ont un taux d'attaque de 0.00 ce sont des ports de services normaux.

### 1.2 Diversité des actions par port

On mesure pour chaque port la variété des durées de flux car un port avec des durées très variables reçoit des types de connexions très différents et peut être le signe possible d'une attaque .

In [13]:
agg_cicids = agg_cicids.drop(columns=['diversite_duree_x', 'diversite_duree_y']) # on supprime les colonnes dupliquées
print(agg_cicids[['Destination Port', 'nb_flux', 'diversite_duree', 'taux_attaque']] #affichage des 10 ports avec le plus fort taux d'attaque
      .nlargest(10, 'taux_attaque')  #tri par taux d'attaque décroissant
      .round(2))

       Destination Port  nb_flux  diversite_duree  taux_attaque
5184              27636        1      44222254.00          1.00
5                    80   136556      44768061.60          0.94
23502             64869        2      31614399.01          0.50
23506             64873        2      31614399.01          0.50
0                     0       54      63678536.54          0.00
1                    21      143      63561088.93          0.00
2                    22      317      63561497.63          0.00
3                    42        3            36.77          0.00
4                    53    30302      51899517.50          0.00
6                    88      170        529218.89          0.00


Ici le port 5184 a un taux d'attaque de 1.00 (100% malveillant) avec une diversité de durée de 44 millions, ce qui signifie des connexions extrêmement variables en durée. Le port 80 confirme son rôle de cible principale avec 0.94 de taux d'attaque et une diversité de 44 millions également. Les ports 23502 et 23506 ont un taux de 0.50 soit moitié attaque et moitié normal donc ce sont probablement des ports utilisés à la fois légitimement et comme vecteur d'attaque. Tous les autres ports ont un taux de 0.00 ce sont donc des ports purement normaux.

### Dataframe final

In [21]:
df_final_cicids = agg_cicids[['Destination Port', 'nb_flux', 'volume_total_octets','duree_moy', 'diversite_duree', 'taux_attaque']]
print("CICIDS : taille", df_final_cicids.shape)
print(df_final_cicids.head().round(2))

CICIDS : taille (23944, 6)
   Destination Port  nb_flux  volume_total_octets    duree_moy  \
0                 0       54                    0  97249154.56   
1                21      143                15230     79837.56   
2                22      317               444838    731020.44   
3                42        3                  702        16.33   
4                53    30302              2378282   1459497.04   

   diversite_duree  taux_attaque  
0      63678536.54           0.0  
1      63561088.93           0.0  
2      63561497.63           0.0  
3            36.77           0.0  
4      51899517.50           0.0  


## Dataset 2 : UNSW-NB15



### 2.1 Agrégation par IP source


In [14]:
agg_unsw = df_unsw.groupby('srcip').agg(
    nb_connexions        = ('srcip', 'count'),              # nombre total de connexions
    nb_dst_uniques       = ('dstip', 'nunique'),            # nombre de destinations uniques
    nb_ports_uniques     = ('dsport', 'nunique'),           # nombre de ports distincts contactés
    nb_proto_uniques     = ('proto', 'nunique'),            # diversité des protocoles utilisés
    volume_envoye        = ('sbytes', 'sum'),               # volume total envoyé en octets
    volume_recu          = ('dbytes', 'sum'),               # volume total reçu en octets
    duree_totale         = ('dur', 'sum'),                  # durée totale des sessions
    duree_moy            = ('dur', 'mean'),                 # durée moyenne par connexion
    duree_max            = ('dur', 'max'),                  # connexion la plus longue
    taux_attaque         = ('Label', 'mean')                # proportion de connexions malveillantes
).reset_index()
print(f"Nombre d'IP distinctes : {len(agg_unsw)}")
print(agg_unsw.sort_values('nb_connexions', ascending=False).head(10).round(2))

Nombre d'IP distinctes : 40
         srcip  nb_connexions  nb_dst_uniques  nb_ports_uniques  \
32  59.166.0.2          67209              10             18986   
30  59.166.0.0          67128              10             18720   
35  59.166.0.5          67091              10             18541   
34  59.166.0.4          66722              10             18574   
31  59.166.0.1          66587              10             18604   
33  59.166.0.3          66145              10             18383   
36  59.166.0.6          64689              10             18346   
38  59.166.0.8          64640              10             18198   
39  59.166.0.9          64187              10             18081   
37  59.166.0.7          63725              10             17769   

    nb_proto_uniques  volume_envoye  volume_recu  duree_totale  duree_moy  \
32                 2      311305266   3511567373      46044.61       0.69   
30                 2      314552145   3433283963      42190.95       0.63   
35 

On obtient 40 IPs distinctes on est donc passer de 700 000 lignes à 40 profils comportementaux complets. Les IP du sous-réseau 59.166.x. dominent avec plus de 63 000 connexions chacune avec 10 destinations uniques et jusqu'à 18 986 ports distincts contactés en effet ce nombre de ports est extrêmement élevé et est la signature typique d'un scan de ports. Chaque IP utilise seulement 2 ou 3 protocoles différents mais transfère des volumes énormes ici jusqu'à 3,5 milliards d'octets reçus. Le taux_attaque à 0.0 pour toutes ces IP dans le top 10 montrent qu'elles sont labellisées comme normales malgré leur comportement intensif.

### 2.2 Features dérivées de l'agrégation

On crée des ratios supplémentaires qui montrent le comportement global de chaque IP.

In [15]:
agg_unsw['ratio_volume'] = agg_unsw['volume_envoye'] / (agg_unsw['volume_recu'] + 1) #ratio envoyé/reçu 
agg_unsw['diversite_dst'] = agg_unsw['nb_dst_uniques'] / agg_unsw['nb_connexions'] #destinations différentes par connexion
agg_unsw['debit_moy'] = (agg_unsw['volume_envoye'] + agg_unsw['volume_recu']) / (agg_unsw['duree_totale'] + 0.001) #débit moyen en octets par seconde 
print(agg_unsw[['srcip', 'ratio_volume', 'diversite_dst', 'debit_moy', 'taux_attaque']].nlargest(10, 'taux_attaque').round(2))

           srcip  ratio_volume  diversite_dst  debit_moy  taux_attaque
26  175.45.176.1          2.34            0.0   13939.58          0.93
27  175.45.176.2          1.22            0.0   29925.49          0.81
25  175.45.176.0          1.46            0.0   11357.04          0.69
28  175.45.176.3          1.07            0.0   18827.48          0.60
0    10.40.170.2      42964.00            0.0      19.89          0.00
1    10.40.182.1          5.53            0.0       6.44          0.00
2    10.40.182.3      42964.00            0.0      19.89          0.00
3     10.40.85.1          5.64            0.0      10.85          0.00
4    10.40.85.30          1.64            0.0      10.80          0.00
5      127.0.0.1          1.46            1.0   98911.97          0.00


On remarque que les IP du sous-réseau 175.45.176. ont les taux d'attaque les plus élevés avec 0.93 pour 175.45.176.1 et 0.81 pour 175.45.176.2 et possèdent des débits plutôt  modérés et une diversite_dst à 0.0 cela signifie que ces IP attaquent toujours les mêmes destinations. À l'inverse le sous-réseau 127.0.0.1 a un débit important de 98 911 octets par s avec un taux d'attaque de 0.0 il s'agit du trafic local normal très rapide. Les IPs 10.40.170.2 et 10.40.182.3 ont un ratio_volume de 42 964 donc elles envoient massivement plus qu'elles ne reçoivent, ce qui peut paraître suspect mais labellisé normal ici. 

### Dataframe final

In [22]:
df_final_unsw = agg_unsw[['srcip', 'nb_connexions', 'nb_dst_uniques','volume_envoye', 'duree_moy', 'ratio_volume','diversite_dst', 'debit_moy', 'taux_attaque']]
print("UNSW: taille", df_final_unsw.shape)
print(df_final_unsw.head().round(2))

UNSW: taille (40, 9)
         srcip  nb_connexions  nb_dst_uniques  volume_envoye  duree_moy  \
0  10.40.170.2            874               1          42964       2.47   
1  10.40.182.1           1670               3         417366      45.85   
2  10.40.182.3            874               1          42964       2.47   
3   10.40.85.1           1680               4         422182      27.27   
4  10.40.85.30            888               1         166152      27.86   

   ratio_volume  diversite_dst  debit_moy  taux_attaque  
0      42964.00            0.0      19.89           0.0  
1          5.53            0.0       6.44           0.0  
2      42964.00            0.0      19.89           0.0  
3          5.64            0.0      10.85           0.0  
4          1.64            0.0      10.80           0.0  



## Dataset 3 : Cybersecurity Threat Detection Logs



### 3.1 Agrégation par IP source

In [16]:
agg_logs = df_logs.groupby('source_ip').agg(
    nb_requetes          = ('source_ip', 'count'),                          # nombre total de requêtes
    nb_urls_uniques      = ('request_path', 'nunique'),                     # nombre d'URLs distinctes visitées
    nb_protocoles        = ('protocol', 'nunique'),                         # diversité des protocoles
    nb_actions_uniques   = ('action', 'nunique'),                           # diversité des actions
    volume_total         = ('bytes_transferred', 'sum'),                    # volume total transféré
    volume_moy           = ('bytes_transferred', 'mean'),                   # volume moyen par requête
    taux_blocage         = ('action', lambda x: (x == 'blocked').mean()),   # proportion de requêtes bloquées
    nb_jours_actifs      = ('timestamp', 'nunique'),                        # nombre de jours d'activité
    threat_label         = ('threat_label', lambda x: x.mode()[0])         # label majoritaire de l'IP
).reset_index()
print(f"Nombre d'IPs distinctes : {len(agg_logs)}")
print(agg_logs.sort_values('nb_requetes', ascending=False).head(10).round(2))

Nombre d'IPs distinctes : 354
           source_ip  nb_requetes  nb_urls_uniques  nb_protocoles  \
336     59.211.9.207        18295              222              7   
3    109.106.120.222        18273              219              7   
348      88.72.40.56        18252              224              7   
33    185.225.185.68        18239              217              7   
9     122.63.201.122        18229              227              7   
307   229.140.23.152        18203              223              7   
327    44.137.187.63        18202              219              7   
337    61.72.172.125        18193              222              7   
8    114.207.221.220        18193              225              7   
27    166.19.156.163        18178              221              7   

     nb_actions_uniques  volume_total  volume_moy  taux_blocage  \
336                   2     455243581    24883.50          0.50   
3                     2     455112077    24906.26          0.49   
348      

Ici on passe de 6 millions de lignes à 354 profils d'IP, chaque IP a visité entre 217 et 227 URL uniques sur 7 protocoles différents avec 2 types d'actions (blocked/allowed). Le volume total par IP atteint entre 450 à 460 millions d'octets avec un volume moyen de 25 000 octets par requête. Toutes les IPs sont actives 365 jours ce qui représente la durée de session complète. Mais comme le dataset est synthétique le taux_blocage est identique à 0.50 pour toutes les IP et le threat_label est bénin pour toutes dans le top 10.

### 3.2 Features dérivées de l'agrégation


In [17]:
agg_logs['ratio_urls_uniques'] = agg_logs['nb_urls_uniques'] / agg_logs['nb_requetes'] #ratio URL uniques par nombre de requêtes
agg_logs['req_par_jour'] = agg_logs['nb_requetes'] / (agg_logs['nb_jours_actifs'] + 1) #nombre moyen de requêtes par jour
print(agg_logs[['source_ip', 'ratio_urls_uniques', 'req_par_jour', 'taux_blocage', 'threat_label']].sort_values('taux_blocage', ascending=False).head(10).round(2))

           source_ip  ratio_urls_uniques  req_par_jour  taux_blocage  \
42     192.168.1.103                0.00         45.08          0.51   
127    192.168.1.180                0.00         44.87          0.51   
115     192.168.1.17                0.00         45.11          0.51   
197    192.168.1.243                0.00         44.99          0.51   
173    192.168.1.221                0.00         44.97          0.51   
68     192.168.1.127                0.00         45.09          0.51   
95     192.168.1.151                0.00         45.11          0.51   
164    192.168.1.213                0.00         45.00          0.51   
315  245.200.237.136                0.01         49.01          0.51   
341     68.36.64.191                0.01         48.91          0.51   

    threat_label  
42        benign  
127       benign  
115       benign  
197       benign  
173       benign  
68        benign  
95        benign  
164       benign  
315       benign  
341       benign 

Le ratio_urls_uniques est à 0.00 pour presque toutes les IPs cela signifie que chaque IP visite presque toujours les mêmes URL. Le req_par_jour est très homogène autour de 45 requêtes par jour pour toutes les IP et le taux_blocage est encore une fois pareil à 0.51 partout. Toutes les IP sont labellisées bénignes dans ce top 10. Le dataset étant synthétique il ne permet pas de distinguer un comportement malveillant d'un comportement normal.

### Dataframe final

In [23]:
df_final_logs = agg_logs[['source_ip', 'nb_requetes', 'nb_urls_uniques','volume_total', 'nb_jours_actifs', 'taux_blocage','ratio_urls_uniques', 'req_par_jour', 'threat_label']]
print("Logs: taille", df_final_logs.shape)
print(df_final_logs.head().round(2))

Logs: taille (354, 9)
         source_ip  nb_requetes  nb_urls_uniques  volume_total  \
0   103.172.167.96        18126              218     451910345   
1     103.93.57.36        17867              222     446793548   
2   104.187.87.205        17817              222     445031851   
3  109.106.120.222        18273              219     455112077   
4    109.153.17.99        18060              219     450150059   

   nb_jours_actifs  taux_blocage  ratio_urls_uniques  req_par_jour  \
0              365          0.50                0.01         49.52   
1              365          0.50                0.01         48.82   
2              365          0.50                0.01         48.68   
3              365          0.49                0.01         49.93   
4              365          0.50                0.01         49.34   

  threat_label  
0       benign  
1       benign  
2       benign  
3       benign  
4       benign  



## Sauvegarde des dataframes agrégés

In [ ]:
agg_cicids.to_csv('../Data/cicids_agregee.csv', index=False)
agg_unsw.to_csv('../Data/unsw_agregee.csv',index=False)
agg_logs.to_csv('../Data/logs_agregee.csv',index=False)